<!-- beginner-banner-v2 -->

> 🧭 <strong>비개발자 수강생 안내</strong> — 이 노트북에서 새로 배우는 것: RAG 5단계(로딩→청킹→임베딩→인덱싱→질의) 를 LlamaIndex 로 직접 조립.
>
> - 📖 강의 페이지: <a href="https://siapapa.github.io/courses/ai-sql-agent/day1/05-llamaindex-intro/" target="_blank" rel="noopener noreferrer">day1/05-llamaindex-intro</a>
> - 🆕 처음이라면 → <a href="https://siapapa.github.io/courses/ai-sql-agent/beginners-guide/" target="_blank" rel="noopener noreferrer">비개발자 학습 가이드</a>
> - 🔤 모르는 단어 → <a href="https://siapapa.github.io/courses/ai-sql-agent/appendix/glossary/" target="_blank" rel="noopener noreferrer">용어 사전</a>
> - 🛠️ 환경/접속 막힘 → <a href="https://siapapa.github.io/courses/ai-sql-agent/setup/" target="_blank" rel="noopener noreferrer">사전 준비</a> · <a href="https://siapapa.github.io/courses/ai-sql-agent/appendix/troubleshooting/" target="_blank" rel="noopener noreferrer">트러블슈팅</a>
>
> 외부 링크는 새 탭으로 열리도록 설정돼 있어 Colab 의 리디렉션 경고 페이지를 거치지 않습니다.<br/>
> <strong>셀은 위에서 아래로 차례대로 실행</strong>하세요. 시연용 코드(<code>구경만 하세요</code> 표시)는 지금 이해 못 해도 100% 정상입니다.

---



# 04. LlamaIndex 파이프라인 개론
> Day 1 · 5H · 소요 약 50분

## 학습 목표

- LlamaIndex의 **Document → Node → Index → Query Engine** 파이프라인을 이해한다.
- `Settings` 전역 설정 방식을 사용한다.
- `SentenceSplitter` / `TokenTextSplitter` 청킹 전략을 비교한다.
- `VectorStoreIndex` 로 기본 RAG 쿼리를 수행한다.

> 이 노트북은 **DB를 쓰지 않습니다.** 인메모리 벡터 인덱스로 문서 RAG만 체험합니다.

In [ ]:
%pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai openai

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """환경변수 `key` 를 채워 넣는다. Colab Secrets → getpass 입력 순으로 시도."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore  (Colab 전용)
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# 이 노트북은 임베딩/LLM 호출이 핵심이므로 OpenAI 키만 필수.
_load_secret("OPENAI_API_KEY", required=True)

print("Environment ready.")

## RAG란?

```
질문 → 관련 문서 검색(top-K) → 질문 + 문서를 LLM에 같이 전달 → 답변
```

한 줄 요약: **"LLM에게 오픈북 시험을 보게 하는 것."**

### LlamaIndex 5단계

```
Documents  →  Nodes (chunks)  →  Embeddings  →  Index  →  QueryEngine
 (원본 문서)    (작은 조각)        (벡터)         (저장/검색)   (질의 응답)
```

In [ ]:
# LlamaIndex 의 `Settings` 는 "이 노트북에서 쓸 LLM/임베딩 모델을 한 번에 정해 두는 전역 설정" 입니다.
# 이 두 줄을 미리 잡아 두면 이후 VectorStoreIndex.from_documents(...) 같은 모든 호출이 동일 모델을 사용합니다.
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# LLM (Large Language Model) — 답변 텍스트 생성용
#   - temperature=0 : 매번 가능한 한 같은 답이 나오도록 (실습 재현성 확보)
#   - max_tokens=1024 : 답변이 너무 길어져 비용이 폭증하는 일을 방지
Settings.llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=1024,
)

# 임베딩 모델 — "텍스트 → 숫자 벡터(좌표)" 로 바꿔서 의미 유사도를 잴 수 있게 한다.
# text-embedding-3-small 은 가격/품질 균형이 좋아 학습용에 적합합니다.
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

print(f"LLM:       {Settings.llm.model}")
print(f"Embedding: {Settings.embed_model.model_name}")

## Document 로딩

LlamaIndex는 PDF, CSV, 웹페이지, SQL 결과 등 다양한 소스에서 `Document` 객체를 만들 수 있습니다. 본 실습에서는 **텍스트에서 직접** 만들어 내부 흐름을 빠르게 훑습니다.

In [ ]:
# Document = LlamaIndex 가 다루는 가장 기본 단위. (텍스트 본문 + 메타데이터) 로 구성됩니다.
# 실전에서는 PDF/CSV/웹페이지에서 자동 추출해 만들지만, 학습용으로 직접 4개를 작성합니다.
from llama_index.core import Document

# 리스트 컴프리헨션이 아니라 명시적인 list 로 작성 — 학생이 한눈에 보고 수정하기 좋게.
hospital_docs = [
    Document(
        # text= 부분이 임베딩/검색 대상.
        # 일부러 줄바꿈/들여쓰기를 그대로 둬서 청킹 시 문장 경계가 어떻게 잡히는지 보기 좋게 했습니다.
        text="""
        서울중앙병원 진료 안내

        진료 시간: 평일 09:00-18:00, 토요일 09:00-13:00
        점심 시간: 12:30-13:30
        응급실: 24시간 운영

        외래 진료 예약은 전화(02-1234-5678) 또는 온라인으로 가능합니다.
        초진 환자는 신분증을 지참해 주세요.
        """,
        # metadata 는 "원문 어디에서 왔는지" 같은 부가 정보. 검색 결과 표기·필터링에 사용됩니다.
        metadata={"source": "hospital_guide", "section": "진료안내", "doc_type": "guide"},
    ),
    Document(
        text="""
        진료과 소개

        내과: 심장, 호흡기, 소화기 질환을 전문으로 합니다.
             김철수, 이영희, 신민아 전문의가 진료합니다.
        외과: 일반외과, 흉부외과, 혈관외과를 운영합니다.
             박민수, 정수진, 권혁준 전문의가 근무합니다.
        소아과: 소아청소년과와 신생아과로 구성되어 있으며,
               최동현, 강미래, 문서영 전문의가 있습니다.
        정형외과: 척추, 관절 질환을 전문으로 하며,
                 윤성호, 한지은 전문의가 진료합니다.
        """,
        metadata={"source": "hospital_guide", "section": "진료과소개", "doc_type": "guide"},
    ),
    Document(
        text="""
        입원 안내

        입원 절차:
        1. 담당 의사의 입원 결정
        2. 원무과에서 입원 수속 (보험증, 신분증 필요)
        3. 병동 배정 및 입실

        병실 종류:
        - 1인실: 250,000원/일
        - 2인실: 150,000원/일
        - 4인실: 80,000원/일
        - 다인실: 건강보험 적용

        면회 시간: 매일 18:00-20:00
        """,
        metadata={"source": "hospital_guide", "section": "입원안내", "doc_type": "guide"},
    ),
    Document(
        text="""
        자주 묻는 질문 (FAQ)

        Q: 진료비 수납은 어떻게 하나요?
        A: 진료 후 1층 수납 창구 또는 무인 수납기를 이용해 주세요.
           카드, 현금, 계좌이체 가능합니다.

        Q: 진단서 발급은 어떻게 하나요?
        A: 1층 제증명 창구에서 신청하실 수 있습니다.
           신분증 지참 필수이며, 발급 소요 시간은 약 30분입니다.

        Q: 주차 요금은 얼마인가요?
        A: 외래 환자 3시간 무료, 이후 30분당 1,000원입니다.
           입원 환자 보호자는 1일 5,000원입니다.
        """,
        metadata={"source": "hospital_guide", "section": "FAQ", "doc_type": "faq"},
    ),
]

print(f"Loaded {len(hospital_docs)} documents")
# 각 Document 의 처음 50자만 미리보기 — `[:50]` 은 문자열 슬라이싱.
for d in hospital_docs:
    print(f"  - [{d.metadata['section']}] {d.text.strip()[:50]}...")

## 청킹 (Node 생성)

하나의 문서를 그대로 임베딩하면 **관련 없는 부분까지 같이 검색**됩니다. 그래서 `chunk_size` 단위로 쪼개어 개별 벡터로 저장합니다.

- **`SentenceSplitter`**: 문장 경계를 최대한 존중 (권장 기본값)
- **`TokenTextSplitter`**: 토큰 수로 엄격히 자름
- `chunk_overlap` — 인접 청크가 겹치는 토큰 수로 문맥을 보존

In [ ]:
# Document → Node 변환 (= "청킹", chunking)
# 큰 문서를 통째로 임베딩하면 한 벡터가 너무 많은 주제를 포함해 검색 정확도가 떨어집니다.
# 그래서 작은 조각(=Node) 으로 자르고 각 조각을 따로 벡터로 만듭니다.
from llama_index.core.node_parser import SentenceSplitter

# chunk_size : 한 청크의 최대 토큰 수. 작을수록 정밀하지만 청크 수가 늘어 비용 증가.
# chunk_overlap : 인접 청크가 겹치는 토큰 수 — 문맥이 잘리는 것을 막아 줌.
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=30)

# 4개 Document → 여러 개 Node 로 분할.
nodes = splitter.get_nodes_from_documents(hospital_docs)

print(f"Generated {len(nodes)} nodes (chunks)\n")
# enumerate 는 인덱스와 값을 함께 돌려준다 — 0번부터 시작이니 i+1 로 1번부터 출력.
for i, node in enumerate(nodes):
    print(f"--- Node {i+1} ({len(node.text)} chars) ---")
    print(f"  meta: {node.metadata}")
    print(f"  text: {node.text.strip()[:100]}...\n")

In [ ]:
# 같은 문서를 토큰 기준으로 자르는 splitter 와 비교.
# - SentenceSplitter : 문장 경계를 최대한 존중 → 사람이 읽기 자연스러운 청크.
# - TokenTextSplitter : 토큰 수를 엄격히 지킴 → 비용이 예측 가능하지만 문장이 중간에 잘릴 수 있음.
from llama_index.core.node_parser import TokenTextSplitter

token_splitter = TokenTextSplitter(chunk_size=256, chunk_overlap=30)
token_nodes = token_splitter.get_nodes_from_documents(hospital_docs)

print(f"SentenceSplitter:  {len(nodes)} nodes")
print(f"TokenTextSplitter: {len(token_nodes)} nodes")
print("\n두 splitter가 만드는 경계 차이를 비교해 보세요.")

## 인덱싱 + Query Engine

`VectorStoreIndex.from_documents(...)` 는 내부에서 다음을 수행합니다:

1. `transformations` 로 Document를 Node로 쪼개고
2. 각 Node를 임베딩 벡터로 변환해
3. 인메모리 벡터스토어에 저장

이후 `as_query_engine()` 으로 질의하면 검색+생성이 한 번에 실행됩니다.

In [ ]:
# VectorStoreIndex — Document 를 받아서 (1) 청킹 → (2) 임베딩 → (3) 인메모리 저장까지 한 번에.
# 이 한 줄이 보통 RAG 튜토리얼의 "마법" 으로 보이는 부분이지만, 실제로는 위에서 본 단계의 자동화입니다.
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    hospital_docs,
    transformations=[splitter],   # 우리가 위에서 만든 SentenceSplitter 를 그대로 재사용
    show_progress=True,           # 진행 막대를 보여 줘 학생들이 임베딩 호출이 일어나는 걸 체감
)
# .as_query_engine() 은 "질문 → 검색 → LLM 답변" 까지 묶어 주는 객체를 만든다.
# similarity_top_k=3 → 가장 가까운 청크 3개만 LLM 컨텍스트에 넣는다 (기본 2).
query_engine = index.as_query_engine(similarity_top_k=3)
print("Index & query engine ready.")

In [ ]:
# 한 번 질문해 보고 "답변 + 출처(어느 청크에서 끌어왔는지)" 를 함께 출력합니다.
# RAG 의 핵심 가치는 "답변" 이 아니라 "그 답변의 근거를 보여 줄 수 있다는 것" 입니다.
response = query_engine.query("내과에는 어떤 의사가 있나요?")
print("Answer:", response.response)

# response.source_nodes : 검색에 사용된 청크 + 유사도 점수
print("\nSources:")
for node in response.source_nodes:
    section = node.metadata.get("section", "?")
    # score 는 0~1 사이 코사인 유사도. 1에 가까울수록 의미가 비슷하다는 뜻.
    print(f"  - [{section}] score={node.score:.3f}")
    print(f"    {node.text.strip()[:80]}...")

In [ ]:
# 추가 질의 세트
questions = [
    "입원 1인실 비용은 얼마인가요?",
    "주차 요금에 대해 알려주세요.",
    "응급실은 언제 이용할 수 있나요?",
]
for q in questions:
    resp = query_engine.query(q)
    top_score = resp.source_nodes[0].score if resp.source_nodes else float("nan")
    print(f"Q: {q}")
    print(f"A: {resp.response}")
    print(f"   (sources={len(resp.source_nodes)}, top={top_score:.3f})\n")

## Retriever — 검색만 수행 (LLM 호출 없이)

생성 없이 "어떤 문서가 질의와 가까운가?" 만 보고 싶을 때 `as_retriever()` 를 사용합니다. LLM 비용 없이 빠르게 검색 품질을 점검할 수 있습니다.

In [ ]:
# Retriever 만 사용 — LLM 호출 없이 "어떤 청크가 질문과 가장 가까운가" 만 본다.
# 검색 품질을 빠르고 싸게 점검할 때 유용합니다 (LLM 비용 ≈ 0).
retriever = index.as_retriever(similarity_top_k=3)
hits = retriever.retrieve("외래 진료 시간이 어떻게 되나요?")

print(f"{len(hits)} hits")
for i, node in enumerate(hits):
    # `:.4f` → 소수 4째 자리까지 출력. 점수의 미세한 차이를 비교할 때 자릿수를 늘리면 도움이 됩니다.
    print(f"\n[{i+1}] score={node.score:.4f} section={node.metadata.get('section','?')}")
    print(f"    {node.text.strip()[:200]}")

## 실습 과제

다음 2 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. chunk_size 비교 실험

`chunk_size`를 128, 256, 512로 바꿔서 각각의 노드 수와 검색 결과를 비교해보세요.

_힌트: `for size in [128, 256, 512]:` 루프 안에서 매번 `SentenceSplitter`와 `VectorStoreIndex`를 새로 만들고, 같은 질문("입원 1인실 비용은?")으로 `query_engine.query()`를 호출해 노드 수와 답변을 출력하세요._

**관찰 포인트:**

- chunk_size가 작을수록 노드 수가 많아지고 검색이 정밀해집니다
- chunk_size가 너무 작으면 문맥이 끊겨서 답변 품질이 떨어질 수 있습니다
- 최적의 chunk_size는 문서 특성과 질문 유형에 따라 다릅니다

### 2. 새 문서 추가 후 재인덱싱

"비급여 항목 안내" 문서를 추가하고 인덱스를 재구성해보세요.

문서에 들어갈 내용 예시:

- 일반 건강검진: 150,000원
- 종합 건강검진: 500,000원
- MRI 촬영: 400,000원~800,000원 (부위별 상이)
- CT 촬영: 200,000원~400,000원
- 도수치료: 1회 80,000원

_힌트: 새 `Document(text=..., metadata={"section": "비급여안내", ...})`를 만들어 `hospital_docs + [new_doc]`로 합친 뒤, `VectorStoreIndex.from_documents(...)`로 재인덱싱하고 "MRI 비용은 얼마인가요?"로 질의하세요._


In [ ]:
# ============================================================
# 실습 과제 — LlamaIndex 파이프라인
# ============================================================

# 실습 1: chunk_size 비교 실험
# TODO: chunk_size를 [128, 256, 512]로 바꿔가며 SentenceSplitter +
#       VectorStoreIndex를 새로 만들고, 같은 질문("입원 1인실 비용은?")으로
#       query_engine.query()를 호출해 노드 수와 답변을 비교하세요.
# 여기에 구현하세요.


# 실습 2: 새 "비급여 항목 안내" 문서 추가 후 재인덱싱
# TODO: Document(text=..., metadata={"section": "비급여안내", ...})를 만들고
#       hospital_docs + [new_doc]로 VectorStoreIndex를 다시 만든 뒤
#       "MRI 비용은 얼마인가요?"로 질의하세요.
# 여기에 구현하세요.


## 다음 노트북에서는…

인덱스는 지금까지 **인메모리** 였습니다. 다음 노트북 **`05_embedding_chromadb.ipynb`** 에서는 임베딩을 직접 호출·시각화하고, ChromaDB 로 벡터를 **디스크에 영속화** 하는 방법을 배웁니다.